# DataFest -  Overview

This notebook showcases a dataset of Wikipedia edits for analysis.

- Ten large Wikipedia language editions: English (`enwiki`), German (`dewiki`), French (`frwiki`), Swedish (`svwiki`), Dutch (`nlwiki`), Russian (`ruwiki`), Spanish (`eswiki`), Italian (`itwiki`), Arabic (`arwiki`), and Polish (`plwiki`).
- Top 10,000 most-read articles in each of those language editions from all of 2025.

There are three kinds of information: 
- page level (information about the page itself)
- page view (how often is the page viewed) and
- page edits (what were the edits made throughout the year).

First the download links and cache functions are declared:

In [1]:
import polars as pl
import json
import urllib.request
from pathlib import Path

def cached(url):
    """Download a file from a URL and cache it locally. Returns the local path, skipping download if already cached."""
    path = Path(url.split("/")[-1])
    if not path.exists():
        print(f"Downloading {path.name}...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(path, "wb") as f:
            f.write(resp.read())
        print(f"Downloaded {path.name}")
    return path

def get_page_edits(language):
    return f"https://analytics.wikimedia.org/published/datasets/one-off/datafete/edit_types/{language}wiki.json.gz"

all_languages = ["ar", "de", "en", "es", "fr", "it", "nl", "pl", "ru", "sv"]

page_data_fn = "https://analytics.wikimedia.org/published/datasets/one-off/datafete/page_info.json.gz"  # 166MB
page_views_fn = "https://analytics.wikimedia.org/published/datasets/one-off/datafete/page_views.json.gz"  # 319MB
page_edits_de_fn = get_page_edits("en")  # 138MB


# Page Level Information

- `wiki_db`: which language version (e.g. `svwiki`)
- `page_id`: for a stable identifier (e.g., `https:<lang>.wikipedia.org/wiki/?curid={page_id}`)
- `qid` as a language-agnostic identifier (in Wikidata) to indicate when articles in separate language editions are about the same topic (`www.wikidata.org/wiki/{qid}`)
- `embedding`: A 50-dimensional topic embedding based on the page's links. These are language-agnostic so comparable across language editions. See [model details](https://meta.wikimedia.org/wiki/Machine_learning_models/Production/Language_agnostic_link-based_article_topic)
- `predicted_labels`: Predicted high-level topics for the article from the same model as above
- `pageviews`: Total number of pageviews in 2025
- `page_title`: the title of the page (for easy interpretability)

In [2]:
page_data = pl.read_ndjson(cached(page_data_fn)) # , n_rows=10000)
print(f"length of page_data: {len(page_data)}")
page_data.head()

length of page_data: 99732


wiki_db,page_id,qid,embedding,predicted_labels,pageviews,page_title
str,i64,str,list[f64],list[struct[2]],i64,str
"""svwiki""",78541,"""Q847769""","[-0.281103, 0.192694, … -0.009869]","[{""Geography.Regions.Europe.Northern_Europe"",1.00001}, {""Geography.Regions.Europe.Europe*"",0.9988405}, … {""Culture.Biography.Biography*"",0.00001}]",21556,"""Filipstad"""
"""svwiki""",12392,"""Q764""","[0.107437, -0.233238, … 0.031688]","[{""STEM.STEM*"",1.00001}, {""STEM.Biology"",0.9987652}, … {""Culture.Media.Music"",0.00001}]",19754,"""Svampar"""
"""svwiki""",34040,"""Q17293""","[-0.147307, 0.18542, … 0.002733]","[{""Culture.Biography.Biography*"",0.930468}, {""Geography.Regions.Asia.Asia*"",0.55448}, … {""Culture.Performing_arts"",0.00001}]",18549,"""Tenzin_Gyatso"""
"""svwiki""",139198,"""Q4939352""","[-0.166814, -0.113468, … -0.097478]","[{""Geography.Regions.Europe.Northern_Europe"",0.990884}, {""Geography.Regions.Europe.Europe*"",0.980886}, … {""STEM.Space"",0.00001}]",60279,"""Jannike_Björling"""
"""svwiki""",55797,"""Q6210414""","[-0.13229, 0.008684, … -0.150917]","[{""Culture.Biography.Biography*"",0.9978273}, {""Culture.Media.Music"",0.9802909}, … {""History_and_Society.Transportation"",0.00001}]",45725,"""Owe_Thörnqvist"""


In [3]:
page_data.filter(page_data['page_id'] == 12561)

wiki_db,page_id,qid,embedding,predicted_labels,pageviews,page_title
str,i64,str,list[f64],list[struct[2]],i64,str
"""enwiki""",12561,"""Q150943""","[-0.174164, 0.1701438, … -0.129535]","[{""Culture.Media.Films"",0.983607}, {""Culture.Media.Media*"",0.973251}, … {""STEM.Computing"",0.00001}]",9989046,"""Gene_Hackman"""


# Page View

- `wiki_db`: which language version (e.g. `svwiki`)
- `page_id`: for a stable identifier (e.g., `https:<lang>.wikipedia.org/wiki/?curid={page_id}`)
- `day`: the actual day formatted as `YYYY-MM-DD`
- `pageviews`: the number of page views for this day

In [4]:
page_views = pl.read_ndjson(cached(page_views_fn)) # , n_rows=10000)
print(f"length of page_views: {len(page_views)}")
page_views.head()

length of page_views: 36249509


wiki_db,page_id,day,pageviews
str,i64,str,i64
"""dewiki""",2365952,"""2025-09-26""",339
"""eswiki""",4715,"""2025-09-20""",2201
"""enwiki""",11715,"""2025-09-23""",4023
"""ruwiki""",18097,"""2025-09-25""",1047
"""frwiki""",7226,"""2025-09-25""",365


# Page Edit Information

Information about all edits to those articles in 2025.
Separate files for each language edition. Files can be rather large (`de` has 138 MB whereas `en` has 1.2 GB ).

- `revision_id`: unique identifier for each edit/revision
- `page_id`: for a stable identifier linking back to the page (e.g., `https:<lang>.wikipedia.org/wiki/?curid={page_id}`)
- `user_text`: the username of the user who made the edit
- `revision_timestamp`: timestamp of when the edit was made (formatted as ISO 8601, e.g. `2025-01-02T09:52:22.000Z`)
- `revision_comment`: comment (if any) the editor added to describe the edit
- `edit_types_json`: computed "edit types" that indicate what was changed by the edit -- see [mwedittypes](https://pypi.org/project/mwedittypes/) library for more details (further details below)
- `revision_tags`: revision tags that are automatically applied. Descriptions can be found at corresponding `https://<lang>.wikipedia.org/wiki/Special:Tags`. Most useful are probably those related to whether an edit was [reverted](https://en.wikipedia.org/wiki/Wikipedia:Reverting) (`mw-reverted`) or itself was reverting another edit (presence of either `mw-undo`, `mw-rollback`, or `mw-manual-revert`)
- `is_bot`: boolean indicating whether the edit was made by a bot account



In [5]:
page_edits_de = pl.read_ndjson(cached(page_edits_de_fn)) # , n_rows=10000)
print(f"length of page_edits_de: {len(page_edits_de)}")
page_edits_de.head()

length of page_edits_de: 1962990


revision_id,page_id,user_text,revision_timestamp,revision_comment,edit_types_json,revision_tags,is_bot
i64,i64,str,str,str,str,list[str],bool
1266511879,1925533,"""Adavid299""","""2025-01-01T00:21:56.000Z""","""/* Departing characters */""","""{""node-edits"": [[""Comment"", ""r…","[""disambiguator-link-added"", ""mobile web edit"", ""mobile edit""]",false
1266520471,77630006,"""Esolo5002""","""2025-01-01T01:16:42.000Z""","""/* Diamond Sports bankruptcy *…","""{""node-edits"": [[""Reference"", …","[""wikieditor""]",false
1266538617,1078,"""Arctic Circle System""","""2025-01-01T03:12:34.000Z""","""/* See also */ Added [[Anti-Ro…","""{""node-edits"": [[""Wikilink"", ""…","[""wikieditor""]",false
1266531416,192892,"""GoingBatty""","""2025-01-01T02:26:56.000Z""","""/* Discography */ fixed link""","""{""node-edits"": [[""Wikilink"", ""…","[""visualeditor""]",false
1266569751,60455274,"""Jmccfip""","""2025-01-01T06:51:28.000Z""","""""","""{""node-edits"": [[""Template"", ""…","[""mobile web edit"", ""mobile edit"", ""visualeditor""]",false


### Details for `edit_types_json`

In [6]:
for revision in page_edits_de.limit(5).to_dicts():
    et = json.loads(revision["edit_types_json"])
    if et["node-edits"]:
        print(f"revision {revision['revision_id']} page {revision['page_id']}")
        for ne in et["node-edits"]:
            print(f"\t{ne}")

revision 1266511879 page 1925533
	['Comment', 'remove', '6: === Departing characters ===', None, [['comment', '=== Returning characters ===\n{| class="wikitable sortable"\n|-\n! scope="col" style="width:190px;" | Character\n! scope="col" style="width:190px;" | Actor(s)\n! scope="col" style="width:10px;"  | {{abbr|Ref(s)|References}}\n|}', None]]]
	['Table', 'insert', '6: === Departing characters ===', None, [['attribute', None, ['class', 'wikitable sortable']], ['cells', 'insert', 6]]]
	['Reference', 'insert', '6: === Departing characters ===', None, []]
	['ExternalLink', 'insert', '6: === Departing characters ===', 'https://www.bbc.com/mediacentre/2025/eastenders-ross-kemp-grant-mitchell-returns-40th-anniversary', [['url', None, 'https://www.bbc.com/mediacentre/2025/eastenders-ross-kemp-grant-mitchell-returns-40th-anniversary']]]
	['Template', 'insert', '6: === Departing characters ===', 'sortname', [['name', None, 'sortname'], ['parameter', None, ['1', 'Ross']], ['parameter', None, [

# Page Views

For each of the following Wikipedia language editions:

`arwiki` · `dewiki` · `enwiki` · `eswiki` · `frwiki` · `itwiki` · `nlwiki` · `plwiki` · `ruwiki` · `svwiki`

Fetch the **top 100 most-viewed articles** and save them as individual CSV files.

## Normalisation

Each article is normalised **independently** on a 0–100 scale to align with Google Trends data:

- **100** = the week with the highest views for that page
- **0** = the week with the lowest views for that page

This means scores are relative to each page's own view history, not compared across pages and these relative hostory will be for the year 2025 alone.

In [7]:
LANGUAGES = ["arwiki", "dewiki", "enwiki", "eswiki", "frwiki", "itwiki", "nlwiki", "plwiki", "ruwiki", "svwiki"]

for lang in LANGUAGES:
    top_100_ids = (
        page_views
        .filter(pl.col('wiki_db') == lang)
        .group_by('page_id')
        .agg(pl.col('pageviews').sum().alias('total_pageviews'))
        .sort('total_pageviews', descending=True)
        .head(100)
        .select('page_id')
    )

    top_100_list = top_100_ids['page_id'].to_list()

    result = (
        page_views
        .filter(pl.col('wiki_db') == lang)
        .filter(pl.col('page_id').is_in(top_100_list))
        .with_columns(
            (pl.col('day').str.to_date('%Y-%m-%d').dt.truncate('1w') - pl.duration(days=1)).alias('week')
        )
        .group_by(['page_id', 'week'])
        .agg(pl.col('pageviews').sum().alias('weekly_views'))
        .with_columns([
            (
                (pl.col('weekly_views') - pl.col('weekly_views').min().over('page_id')) /
                (pl.col('weekly_views').max().over('page_id') - pl.col('weekly_views').min().over('page_id')) * 100
            ).round(0).cast(pl.Int64).alias('normalized_views')
        ])
        .sort(['page_id', 'week'])
    )

    result.write_csv(f'{lang}_normalized.csv')

# 'Predicted_labels' Extraction

For each Wikipedia language edition, filters `page_views` to the relevant wiki, aggregates total views per `page_id`, and takes the **top 100**. Results are joined with `page_data` and saved to `top100_pageviews/{lang}_top100.csv`.

If a `predicted_labels` column is present, label-probability pairs are flattened into a readable string (e.g. `politics:0.812, science:0.143`) before saving. Any remaining nested columns (`List`, `Struct`, `Array`) are dropped. Each result is also stored in memory as `top100_{lang}`.

In [8]:
import os

languages = ["ar", "de", "en", "es", "fr", "it", "nl", "pl", "ru", "sv"]

os.makedirs('top100_pageviews', exist_ok=True)

for lang in languages:
    df = (
        page_views
        .filter(pl.col('wiki_db') == f'{lang}wiki')
        .group_by('page_id')
        .agg(pl.col('pageviews').sum().alias('total_pageviews'))
        .sort('total_pageviews', descending=True)
        .head(100)
        .join(page_data, on='page_id', how='left')
    )
    
    if 'predicted_labels' in df.columns:
        df = df.with_columns(
            pl.col('predicted_labels').list.eval(
                pl.concat_str([
                    pl.element().struct.field('label'),
                    pl.element().struct.field('probability').round(3).cast(pl.String)
                ], separator=':')
            ).list.join(', ')
        )
    
    nested_cols = [col for col in df.columns if str(df[col].dtype).startswith(('List', 'Struct', 'Array'))]
    df = df.drop(nested_cols)
    
    globals()[f'top100_{lang}'] = df
    df.write_csv(f'top100_pageviews/{lang}_top100.csv')
    print(f"Saved {lang} — {len(df)} rows")

print("Done!")

Saved ar — 122 rows
Saved de — 119 rows
Saved en — 103 rows
Saved es — 123 rows
Saved fr — 116 rows
Saved it — 103 rows
Saved nl — 115 rows
Saved pl — 117 rows
Saved ru — 131 rows
Saved sv — 119 rows
Done!


In [ ]:
top_100_ids = (
    page_views
    .filter(pl.col('wiki_db') == 'enwiki')
    .group_by('page_id')
    .agg(pl.col('pageviews').sum().alias('total_pageviews'))
    .sort('total_pageviews', descending=True)
    .head(100)
    .select('page_id')
)

result = (
    page_views
    .filter(pl.col('wiki_db') == 'enwiki')
    .filter(pl.col('page_id').is_in(top_100_ids['page_id']))
    .group_by(['page_id', 'week'])
    .agg(pl.col('pageviews').sum().alias('weekly_views'))
    .with_columns([
        (
            (pl.col('weekly_views') - pl.col('weekly_views').min().over('page_id')) /
            (pl.col('weekly_views').max().over('page_id') - pl.col('weekly_views').min().over('page_id')) * 100
        ).alias('normalized_views')
    ])
    .sort(['page_id', 'week'])
)

# Wiki Normalized Views Pipeline

## Overview
Automatically joins normalized pageview data with top 100 pageview metadata for all available languages.

## How It Works

1. **Auto-detects languages** by scanning all `*_normalized.csv` files in `Normalised_views/`
2. **For each language**, reads the corresponding top 100 file from `top100_pageviews/`
3. **Joins** both datasets on `page_id`
4. **Saves** the result to `Combined_view/`

## Folder Structure

```
project/
├── Normalised_views/
│   ├── arwiki_normalized.csv
│   ├── dewiki_normalized.csv
│   └── ...
├── top100_pageviews/
│   ├── arwiki_top100.csv
│   ├── dewiki_top100.csv
│   └── ...
└── Combined_view/          ← output goes here
```

## Output
One combined CSV per language, e.g. `Combined_view/arwiki_combined.csv`

In [ ]:
import polars as pl
import glob

all_files = glob.glob('Normalised_views/*_normalized.csv')

for filepath in all_files:
    lang = filepath.split('/')[-1].replace('_normalized.csv', '')  # e.g. 'arwiki'

    normalized = pl.read_csv(f'Normalised_views/{lang}_normalized.csv')
    top100 = pl.read_csv(f'top100_pageviews/{lang}_top100.csv').filter(pl.col('wiki_db') == lang)

    combined = normalized.join(top100, on='page_id', how='inner')
    combined.write_csv(f'Combined_view/{lang}_combined.csv')

## Google Trends + Wikipedia Pageviews Join

For each language, weekly **Wikipedia pageview** data is joined with **Google Trends** search interest on `page_title` and `Time`.
The output is a single CSV per language containing both signals aligned by week, ready for analysis.

In [ ]:
import polars as pl
import os
import glob


main = pl.read_csv('Combined_view/enwiki_comb_normalized.csv')

all_files = glob.glob('google_trends_cache_en_100_rescaled/*.csv')

dfs = []
for filepath in all_files:
   filename = os.path.basename(filepath).replace('.csv', '')
   df = pl.read_csv(filepath)
   df = df.with_columns(pl.lit(filename).alias('page_title'))
   dfs.append(df)

all_data = pl.concat(dfs)

result = main.join(all_data, on=['page_title', 'Time'], how='left')
result.write_csv('Combined_view/enwiki_comb_normalized_with_trends.csv')